In [ ]:
!pip -q install gradio ultralytics rembg onnxruntime opencv-python torch torchvision pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import io
import random
import numpy as np
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torchvision import models, transforms

from ultralytics import YOLO
import gradio as gr

from rembg import remove

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#Model paths in google drive
VGG_PATH  = "/content/drive/MyDrive/trained_vgg19_half_split.pth"
SLT_PATH  = "/content/drive/MyDrive/sltunet_final_best.pth"
YOLO_PATH = "/content/drive/MyDrive/best.pt"

#Background images folder
ENV_BG_DIR = "/content/drive/MyDrive/background_images"

#Class names used during training
class_names = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','del','nothing','space'
]
num_classes = len(class_names)
print("Number of classes:", num_classes)

In [ ]:
#VGG19 loader
def load_vgg19():
    model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
    for p in model.features.parameters():
        p.requires_grad = False
    model.classifier[6] = nn.Linear(4096, num_classes)
    state_dict = torch.load(VGG_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    return model


#SLTUNET definition
class SLTUNet_Pretrained(nn.Module):
    def __init__(self, num_classes=29):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        self.enc1 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.enc2 = base.layer1
        self.enc3 = base.layer2
        self.enc4 = base.layer3
        self.enc5 = base.layer4

        self.reduce = nn.Conv2d(2048, 512, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        x = self.enc5(x)
        x = self.reduce(x)
        x = self.pool(x)
        return self.head(x)

def load_sltunet():
    model = SLTUNet_Pretrained(num_classes=num_classes)
    ckpt = torch.load(SLT_PATH, map_location=device)

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    return model


#YOLOv8 loader
def load_yolo():
    return YOLO(YOLO_PATH)

vgg_model = load_vgg19()
slt_model = load_sltunet()
yolo_model = load_yolo()
print("All models loaded successfully.")

In [ ]:
#Image processing and transform

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def predict_pytorch(model, pil_img):
    x = tf(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), probs

def predict_yolo(pil_img):
    img = np.array(pil_img.convert("RGB"))
    result = yolo_model(img, imgsz=224, verbose=False)[0]
    idx = int(result.probs.top1)
    conf = float(result.probs.top1conf)
    pred_name = result.names[idx]
    return pred_name, conf

In [ ]:
def remove_background_and_crop(pil_img, padding=20):
    """
    1. Removes background using rembg
    2. Finds the non-transparent bounding box
    3. Crops the visible foreground tightly
    4. Returns cropped RGBA image
    """

    #Ensuring RGB input
    pil_img = pil_img.convert("RGB")

    #Removing background
    rgba = remove(pil_img).convert("RGBA")

    #Getting alpha channel and finding visible bbox
    alpha = rgba.getchannel("A")
    bbox = alpha.getbbox()
    if bbox is None:
        w, h = pil_img.size
        crop_w, crop_h = int(w * 0.65), int(h * 0.65)
        left = (w - crop_w) // 2
        top = (h - crop_h) // 2
        return pil_img.crop((left, top, left + crop_w, top + crop_h)).convert("RGBA")

    x1, y1, x2, y2 = bbox

    #Padding
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(rgba.width, x2 + padding)
    y2 = min(rgba.height, y2 + padding)

    cropped = rgba.crop((x1, y1, x2, y2))
    return cropped

In [ ]:
def paste_on_solid_background(fg_rgba, bg_hex="#00FF00", out_size=(224, 224), pad=0.08):
    color_rgb = tuple(int(bg_hex[i:i+2], 16) for i in (1, 3, 5))
    bg = Image.new("RGB", out_size, color_rgb)

    W, H = out_size
    max_w = int(W * (1 - 2 * pad))
    max_h = int(H * (1 - 2 * pad))

    fw, fh = fg_rgba.size
    scale = min(max_w / fw, max_h / fh)
    new_w = max(1, int(fw * scale))
    new_h = max(1, int(fh * scale))

    fg_resized = fg_rgba.resize((new_w, new_h), Image.LANCZOS)

    x = (W - new_w) // 2
    y = (H - new_h) // 2

    bg_rgba = bg.convert("RGBA")
    bg_rgba.alpha_composite(fg_resized, (x, y))

    return bg_rgba.convert("RGB")


def list_env_backgrounds():
    if not os.path.isdir(ENV_BG_DIR):
        return []
    files = []
    for f in os.listdir(ENV_BG_DIR):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            files.append(os.path.join(ENV_BG_DIR, f))
    return sorted(files)


def paste_on_environment_background(fg_rgba, env_path, out_size=(224, 224), pad=0.08):
    bg = Image.open(env_path).convert("RGB").resize(out_size)

    W, H = out_size
    max_w = int(W * (1 - 2 * pad))
    max_h = int(H * (1 - 2 * pad))

    fw, fh = fg_rgba.size
    scale = min(max_w / fw, max_h / fh)
    new_w = max(1, int(fw * scale))
    new_h = max(1, int(fh * scale))

    fg_resized = fg_rgba.resize((new_w, new_h), Image.LANCZOS)

    x = (W - new_w) // 2
    y = (H - new_h) // 2

    bg_rgba = bg.convert("RGBA")
    bg_rgba.alpha_composite(fg_resized, (x, y))

    return bg_rgba.convert("RGB")


def parse_color(color_value):

    if isinstance(color_value, tuple) or isinstance(color_value, list):
        return tuple(color_value[:3])

    color_value = str(color_value).strip()

    #hex color
    if color_value.startswith("#") and len(color_value) == 7:
        return tuple(int(color_value[i:i+2], 16) for i in (1, 3, 5))

    #rgb or rgba
    if color_value.startswith("rgb"):
        nums = color_value[color_value.find("(")+1:color_value.find(")")].split(",")
        nums = [int(float(x.strip())) for x in nums[:3]]
        return tuple(nums)

    #white
    return (255, 255, 255)


def paste_on_solid_background(fg_rgba, bg_hex="#00FF00", out_size=(224, 224), pad=0.08):
    color_rgb = parse_color(bg_hex)
    bg = Image.new("RGB", out_size, color_rgb)

    W, H = out_size
    max_w = int(W * (1 - 2 * pad))
    max_h = int(H * (1 - 2 * pad))

    fw, fh = fg_rgba.size
    scale = min(max_w / fw, max_h / fh)
    new_w = max(1, int(fw * scale))
    new_h = max(1, int(fh * scale))

    fg_resized = fg_rgba.resize((new_w, new_h), Image.LANCZOS)

    x = (W - new_w) // 2
    y = (H - new_h) // 2

    bg_rgba = bg.convert("RGBA")
    bg_rgba.alpha_composite(fg_resized, (x, y))

    return bg_rgba.convert("RGB")

In [ ]:
def run_demo(image, model_name, mode, bg_color, env_bg_path):
    if image is None:
        return None, "Please upload an image!"

    try:
        #Extracting foreground
        fg_crop = remove_background_and_crop(image, padding=20)

        #Hand flip
        if mode == "Hand Flip":
            fg_crop = ImageOps.mirror(fg_crop)

        #Composing final image
        if mode == "Solid Background":
            final_img = paste_on_solid_background(fg_crop, bg_hex=bg_color, out_size=(224, 224))
        elif mode == "Environment Background":
            if env_bg_path is None or env_bg_path == "":
                return None, "Please select an environment background!"
            final_img = paste_on_environment_background(fg_crop, env_bg_path, out_size=(224, 224))
        elif mode == "Hand Flip":
            #Putting flipped hand on white background
            final_img = paste_on_solid_background(fg_crop, bg_hex="#FFFFFF", out_size=(224, 224))
        else:
            #White background
            final_img = paste_on_solid_background(fg_crop, bg_hex="#FFFFFF", out_size=(224, 224))

        #Model prediction
        if model_name == "SLTUNET":
            pred, conf, _ = predict_pytorch(slt_model, final_img)
        elif model_name == "VGG19":
            pred, conf, _ = predict_pytorch(vgg_model, final_img)
        else:
            pred, conf = predict_yolo(final_img)

        text = (
            f"**Model:** {model_name}\n\n"
            f"**Background Mode:** {mode}\n\n"
            f"**Prediction:** {pred}\n\n"
            f"**Confidence:** {conf:.4f}"
        )
        return final_img, text

    except Exception as e:
        return None, f"Error: {str(e)}"

In [ ]:
env_list = list_env_backgrounds()
env_default = env_list[0] if len(env_list) > 0 else ""

with gr.Blocks(title="ASL Alphabet Robustness Demo") as demo:
    gr.Markdown("# Interactive Interface Demo - ASL")
    gr.Markdown(
        "Upload a hand-sign image. The tool removes the original background from the image, "
        "crops the gesture region, optionally flips it, places it onto a chosen new background, "
        "and then runs prediction using one of the trained models."
    )

    with gr.Row():
        inp = gr.Image(type="pil", label="Upload Image", sources=["upload"])
        out_img = gr.Image(type="pil", label="Processed Image")

    with gr.Row():
        model_dd = gr.Dropdown(
            ["SLTUNET", "YOLOv8", "VGG19"],
            value="SLTUNET",
            label="Model"
        )
        mode_dd = gr.Dropdown(
            ["White Background", "Solid Background", "Environment Background", "Hand Flip"],
            value="Solid Background",
            label="Background Mode"
        )

    with gr.Row():
        color_picker = gr.ColorPicker(value="#00FF00", label="Solid Background Color")
        env_dropdown = gr.Dropdown(
            env_list,
            value=env_default,
            label="Environment Background Image"
        )

    run_btn = gr.Button("Run")
    output_text = gr.Markdown()

    run_btn.click(
        fn=run_demo,
        inputs=[inp, model_dd, mode_dd, color_picker, env_dropdown],
        outputs=[out_img, output_text]
    )

demo.launch(share=True)